# Notebook 10 — Financial and Governance Anomaly Detection

**Purpose:** Identify potentially suspicious financial and governance patterns for investigator triage using peer comparisons, historical changes, and unsupervised anomaly detection. **Anomalies are not evidence of fraud.**

**Scope:** Primary 2024 full Form 990 population (known-label cohort as established in 09D). Form 990-EZ is separately inventoried, **not** scored using incompatible fields. Self-reported diversion labels are used only for *retrospective diagnostic checks*, never to construct anomaly scores.

**Outputs:** `data/processed/notebook10/` (peer benchmarks, indicators, ranked queue, QA summaries, metadata). Does not overwrite Notebook 09 artifacts. Run from the project root or `notebooks/` using the `irs990` environment.

**Revision:** Directional adverse indicators, minimum four-indicator primary coverage, separate exploratory cohort, and original-versus-revised ranking diagnostics. Statistical flags are not evidence of fraud.

## 1. Setup, input validation, and reproducibility

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
INPUT = ROOT / 'data/processed/nccs_longitudinal_features_v1.parquet'
COHORTS = ROOT / 'data/processed/notebook09d/03_2024_scoring_cohorts.parquet'
OUT = ROOT / 'data/processed/notebook10'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
MIN_PEER_SIZE = 100
FIT_SAMPLE = 100_000
USE_ISOLATION_FOREST = True
assert INPUT.exists(), f'Missing input: {INPUT}'
assert COHORTS.exists(), 'Run Notebook 09D first to generate scoring cohorts.'

def sqlpath(p):
    return str(p).replace("'", "''")

con = duckdb.connect()
con.execute('SET threads = 4')
con.execute("SET memory_limit = '6GB'")
con.execute(f"CREATE VIEW features AS SELECT * FROM read_parquet('{sqlpath(INPUT)}')")
con.execute(f"CREATE VIEW cohorts AS SELECT * FROM read_parquet('{sqlpath(COHORTS)}')")
cols = set(con.execute('DESCRIBE features').df().column_name)
required = {'OBJECTID','ein','tax_year','fraud_label','total_assets','total_liabilities',
            'total_revenue','total_expenses','net_income','debt_to_asset_ratio',
            'voting_members','independent_members','pct_independent_board',
            'rev_growth_3yr','asset_growth_3yr','rev_volatility_3yr',
            'flag_sustained_deficit','flag_board_independence_drop','historical_filing_depth'}
assert required.issubset(cols), f'Missing features: {required-cols}'
print('Input:', INPUT)
print('Output:', OUT)
print('2024 rows:', con.execute('SELECT COUNT(*) FROM features WHERE tax_year=2024').fetchone()[0])

## 2. Confirm filing-type eligibility and source coverage

The 09D cohort file identifies known vs. unknown disclosure-label populations. Its relationship to Form 990/990-EZ was established in 09D. This notebook **does not assume missing means zero**, and does not claim that 990-EZ organizations are safe or risky. Check joins and label availability before scoring.

In [ ]:
audit = con.execute("""
SELECT c.cohort, c.eligible_for_labeled_validation AS eligible,
       COUNT(*) AS filings,
       COUNT(*) FILTER (WHERE f.OBJECTID IS NULL) AS unmatched,
       COUNT(*) FILTER (WHERE f.fraud_label=1) AS reported_diversions,
       COUNT(*) FILTER (WHERE f.total_revenue IS NULL) AS missing_revenue,
       COUNT(*) FILTER (WHERE f.total_assets IS NULL) AS missing_assets
FROM cohorts c LEFT JOIN features f ON c.OBJECTID=f.OBJECTID AND f.tax_year=2024
GROUP BY 1,2 ORDER BY 1,2
""").df()
display(audit)
assert audit.unmatched.sum()==0
assert audit.filings.sum()==con.execute('SELECT COUNT(*) FROM features WHERE tax_year=2024').fetchone()[0]
assert con.execute('SELECT COUNT(*) FROM cohorts').fetchone()[0] == con.execute('SELECT COUNT(DISTINCT OBJECTID) FROM cohorts').fetchone()[0]
audit.to_csv(OUT/'02_cohort_coverage.csv',index=False)

## 3. Build interpretable anomaly indicators

Indicators include expense/revenue, debt/assets, revenue growth, volatility, board independence, sustained deficits, and independence drops. Ratios with zero or negative denominators are left NULL; negatives and implausible values are retained for separate QA rather than silently clipped. These are **screening indicators**, not legal or accounting determinations.

Peers are based on **revenue size only** for this first version. This is deliberately simple; future peer groups should include nonprofit subsector (e.g., NTEE) if reliably available.

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP VIEW base990 AS
SELECT f.OBJECTID, f.ein, f.tax_year, f.fraud_label,
       TRY_CAST(f.total_revenue AS DOUBLE) AS revenue,
       TRY_CAST(f.total_expenses AS DOUBLE) AS expenses,
       TRY_CAST(f.total_assets AS DOUBLE) AS assets,
       TRY_CAST(f.total_liabilities AS DOUBLE) AS liabilities,
       TRY_CAST(f.net_income AS DOUBLE) AS net_income,
       TRY_CAST(f.voting_members AS DOUBLE) AS voting_members,
       TRY_CAST(f.independent_members AS DOUBLE) AS independent_members,
       TRY_CAST(f.pct_independent_board AS DOUBLE) AS board_independence,
       TRY_CAST(f.rev_growth_3yr AS DOUBLE) AS revenue_growth_3yr,
       TRY_CAST(f.asset_growth_3yr AS DOUBLE) AS asset_growth_3yr,
       TRY_CAST(f.rev_volatility_3yr AS DOUBLE) AS revenue_volatility_3yr,
       TRY_CAST(f.flag_sustained_deficit AS DOUBLE) AS sustained_deficit,
       TRY_CAST(f.flag_board_independence_drop AS DOUBLE) AS board_independence_drop,
       TRY_CAST(f.historical_filing_depth AS DOUBLE) AS historical_filing_depth
FROM features f JOIN cohorts c ON f.OBJECTID=c.OBJECTID
WHERE f.tax_year=2024 AND c.cohort='known_label'
  AND c.eligible_for_labeled_validation=TRUE
""")
con.execute("""
CREATE OR REPLACE TEMP VIEW indicators AS
SELECT *,
 CASE WHEN revenue>0 THEN expenses/revenue END AS expense_revenue_ratio,
 CASE WHEN assets>0 THEN liabilities/assets END AS liabilities_assets_ratio,
 CASE WHEN revenue>0 THEN net_income/revenue END AS net_margin,
 CASE WHEN revenue IS NULL THEN 'unknown'
      WHEN revenue<100000 THEN 'under_100k'
      WHEN revenue<1000000 THEN '100k_to_1m'
      WHEN revenue<10000000 THEN '1m_to_10m'
      WHEN revenue<100000000 THEN '10m_to_100m'
      ELSE '100m_plus' END AS peer_group,
 CASE WHEN revenue<=0 THEN 1 ELSE 0 END AS nonpositive_revenue,
 CASE WHEN assets<=0 THEN 1 ELSE 0 END AS nonpositive_assets,
 CASE WHEN voting_members<=0 THEN 1 ELSE 0 END AS nonpositive_board,
 CASE WHEN board_independence<0 OR board_independence>1 THEN 1 ELSE 0 END AS board_ratio_out_of_range
FROM base990
""")
counts=con.execute('SELECT peer_group,COUNT(*) n FROM indicators GROUP BY 1 ORDER BY 1').df()
display(counts)
assert counts.n.min()>=MIN_PEER_SIZE, 'Peer group too small; regroup before calculating peer percentiles.' 

## 4. Peer-relative indicators and revised risk ranking

Preserve the original two-sided anomaly score for comparison. For the primary review ranking, use **directional, potentially adverse deviations**: high expense/revenue, high liabilities/assets, low net margin, unusually negative growth, high revenue volatility, and low board independence. This is a *review heuristic*, not an empirically validated fraud-risk score. Missing metrics contribute zero to the seven-indicator sum; a minimum of four observed metrics is required for the primary queue. Isolation Forest remains separate.

In [ ]:
metrics = ['expense_revenue_ratio','liabilities_assets_ratio','net_margin',
           'revenue_growth_3yr','asset_growth_3yr','revenue_volatility_3yr','board_independence']
parts=[]
for m in metrics:
    parts.append(f"CASE WHEN {m} IS NOT NULL AND isfinite({m}) THEN CUME_DIST() OVER (PARTITION BY peer_group, ({m} IS NOT NULL AND isfinite({m})) ORDER BY CASE WHEN isfinite({m}) THEN {m} END) END AS pct_{m}")
con.execute('CREATE OR REPLACE TEMP VIEW peer_percentiles AS SELECT *, '+', '.join(parts)+' FROM indicators')
# Financial two-tail extremeness; governance weakness only low independence.
severity = [f'CASE WHEN pct_{m} IS NOT NULL THEN 2*ABS(pct_{m}-0.5) END' for m in metrics[:-1]]
severity += ['CASE WHEN pct_board_independence IS NOT NULL THEN 1-pct_board_independence END']
exprs=[f'{s} AS severity_{m}' for m,s in zip(metrics,severity)]
con.execute('CREATE OR REPLACE TEMP VIEW severity AS SELECT *, '+', '.join(exprs)+' FROM peer_percentiles')
sevcols=[f'severity_{m}' for m in metrics]
num='+'.join(f'COALESCE({m},0)' for m in sevcols)
den='+'.join(f'CASE WHEN {m} IS NOT NULL THEN 1 ELSE 0 END' for m in sevcols)
con.execute(f"""CREATE OR REPLACE TEMP VIEW scored AS SELECT *,
  ({den}) AS available_indicator_count,
  CASE WHEN ({den})>0 THEN ({num})/({den}) END AS peer_anomaly_index
  FROM severity""")
summary=con.execute('SELECT COUNT(*) filings, MIN(available_indicator_count) min_indicators, AVG(available_indicator_count) avg_indicators, MIN(peer_anomaly_index) min_index, MAX(peer_anomaly_index) max_index FROM scored').df()
display(summary)
assert summary.min_index.iloc[0]>=0 and summary.max_index.iloc[0]<=1.000001
con.execute(f"COPY (SELECT peer_group, COUNT(*) AS n, MEDIAN(revenue) AS median_revenue, MEDIAN(expense_revenue_ratio) AS median_expense_ratio, MEDIAN(liabilities_assets_ratio) AS median_debt_ratio, MEDIAN(board_independence) AS median_board_independence FROM scored GROUP BY 1) TO '{sqlpath(OUT/'04_peer_benchmarks.parquet')}' (FORMAT PARQUET)")
# Directional adverse tail severity (0..1). High expenses, leverage and volatility;
# low margins, growth and board independence. Do not label these as proven fraud signals.
directions = {
    'expense_revenue_ratio': 'high',
    'liabilities_assets_ratio': 'high',
    'net_margin': 'low',
    'revenue_growth_3yr': 'low',
    'asset_growth_3yr': 'low',
    'revenue_volatility_3yr': 'high',
    'board_independence': 'low',
}
# CUME_DIST ties are conservative for the low tail (1-pct). High-tail scores
# are not zero for the median; values near the median should not be prioritized.
# A severity of 0 corresponds to the non-adverse half of each peer distribution.
parts=[]
for m,d in directions.items():
    p=f'pct_{m}'
    if d=='high':
        expr=f'GREATEST(0.0, 2.0*({p}-0.5))'
    else:
        expr=f'GREATEST(0.0, 2.0*(0.5-{p}))'
    parts.append(f'CASE WHEN {p} IS NOT NULL THEN {expr} END AS adverse_{m}')
con.execute('CREATE OR REPLACE TEMP VIEW directional AS SELECT *, '+', '.join(parts)+' FROM scored')
adverse_cols=[f'adverse_{m}' for m in metrics]
# Fixed denominator penalizes sparse evidence rather than averaging only extremes.
fixed_sum='+'.join(f'COALESCE({c},0.0)' for c in adverse_cols)
con.execute(f"""CREATE OR REPLACE TEMP VIEW revised AS SELECT *,
    ({fixed_sum})/7.0 AS adverse_peer_index,
    CASE WHEN available_indicator_count >= 4 THEN 'primary_review'
         ELSE 'limited_data' END AS review_cohort,
    ({fixed_sum}) AS adverse_indicator_sum
    FROM directional""")
display(con.execute('SELECT review_cohort,COUNT(*) filings, AVG(available_indicator_count) avg_available, AVG(adverse_peer_index) avg_score FROM revised GROUP BY 1 ORDER BY 1').df())


## 5. Optional Isolation Forest — unsupervised corroborating signal

Fit only on 2024 eligible Form 990 filings, using a reproducible sample. Impute medians based on the training sample, transform with RobustScaler, and score the whole eligible population. No outcome label is supplied. This is a **within-2024 descriptive detector** and must not be interpreted as a prospective or externally validated fraud model.

For easier runtime, set `USE_ISOLATION_FOREST=False`.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
IF_FEATURES=['expense_revenue_ratio','liabilities_assets_ratio','net_margin',
             'revenue_growth_3yr','asset_growth_3yr','revenue_volatility_3yr','board_independence']
if USE_ISOLATION_FOREST:
    frame=con.execute('SELECT OBJECTID, '+','.join(IF_FEATURES)+' FROM scored ORDER BY OBJECTID').df()
    X=frame[IF_FEATURES].replace([np.inf,-np.inf],np.nan)
    valid=X.notna().sum(axis=1)>0
    assert valid.sum()>0
    sample=X.loc[valid].sample(n=min(FIT_SAMPLE,int(valid.sum())),random_state=SEED)
    prep=make_pipeline(SimpleImputer(strategy='median',add_indicator=False),RobustScaler())
    Xfit=prep.fit_transform(sample)
    forest=IsolationForest(n_estimators=150,max_samples=256,contamination='auto',random_state=SEED,n_jobs=4)
    forest.fit(Xfit)
    # Higher values mean more anomalous; leave no-feature rows unscored.
    scores=np.full(len(frame),np.nan)
    batch=50_000
    positions=np.flatnonzero(valid.to_numpy())
    for start in range(0,len(positions),batch):
        ix=positions[start:start+batch]
        scores[ix]=-forest.score_samples(prep.transform(X.iloc[ix]))
    if_out=pd.DataFrame({'OBJECTID':frame.OBJECTID,'isolation_anomaly_score':scores})
    con.register('if_output',if_out)
    con.execute(f"COPY (SELECT * FROM if_output) TO '{sqlpath(OUT/'05_isolation_forest_scores.parquet')}' (FORMAT PARQUET)")
    display(if_out.isolation_anomaly_score.describe().to_frame())
    del frame,X,sample,Xfit
else:
    print('Isolation Forest skipped; peer anomaly index remains available.')

## 6. Generate revised 2024 review queues

The primary queue requires at least four observed peer indicators and ranks by the fixed-denominator directional index. Filings with fewer indicators remain in a separate **limited-data** queue. Preserve the old two-sided anomaly index and its original rank to compare the change. Isolation Forest is a corroborating, not combined, signal.

In [ ]:
join_if = f"LEFT JOIN read_parquet('{sqlpath(OUT/'05_isolation_forest_scores.parquet')}') i ON s.OBJECTID=i.OBJECTID" if USE_ISOLATION_FOREST else ''
if_col='i.isolation_anomaly_score' if USE_ISOLATION_FOREST else 'CAST(NULL AS DOUBLE)'
con.execute(f"""
COPY (
SELECT s.OBJECTID,s.ein,s.tax_year,s.peer_group,
       s.revenue,s.expenses,s.assets,s.liabilities,s.net_income,
       s.expense_revenue_ratio,s.liabilities_assets_ratio,s.net_margin,
       s.revenue_growth_3yr,s.asset_growth_3yr,s.revenue_volatility_3yr,
       s.board_independence,s.sustained_deficit,s.board_independence_drop,
       s.historical_filing_depth,s.available_indicator_count,
       s.peer_anomaly_index, s.adverse_peer_index, s.adverse_indicator_sum,
       s.review_cohort, {if_col} AS isolation_anomaly_score,
       s.nonpositive_revenue,s.nonpositive_assets,s.nonpositive_board,s.board_ratio_out_of_range,
       s.severity_expense_revenue_ratio,s.severity_liabilities_assets_ratio,
       s.severity_net_margin,s.severity_revenue_growth_3yr,
       s.severity_asset_growth_3yr,s.severity_revenue_volatility_3yr,
       s.severity_board_independence,
       s.adverse_expense_revenue_ratio,s.adverse_liabilities_assets_ratio,
       s.adverse_net_margin,s.adverse_revenue_growth_3yr,
       s.adverse_asset_growth_3yr,s.adverse_revenue_volatility_3yr,
       s.adverse_board_independence,
       ROW_NUMBER() OVER (ORDER BY s.peer_anomaly_index DESC NULLS LAST, s.OBJECTID) AS original_anomaly_rank,
       CASE WHEN s.review_cohort='primary_review' THEN
           COUNT(*) FILTER (WHERE s.review_cohort='primary_review') OVER (
               ORDER BY s.adverse_peer_index DESC, s.available_indicator_count DESC, s.OBJECTID
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
       END AS primary_rank
FROM revised s {join_if}
) TO '{sqlpath(OUT/'06_2024_form990_anomaly_queue.parquet')}' (FORMAT PARQUET)
""")
qpath=OUT/'06_2024_form990_anomaly_queue.parquet'
con.execute(f"COPY (SELECT * FROM read_parquet('{sqlpath(qpath)}') WHERE review_cohort='primary_review' ORDER BY primary_rank) TO '{sqlpath(OUT/'06_primary_review_queue.parquet')}' (FORMAT PARQUET)")
con.execute(f"COPY (SELECT * FROM read_parquet('{sqlpath(qpath)}') WHERE review_cohort='limited_data' ORDER BY adverse_peer_index DESC, OBJECTID) TO '{sqlpath(OUT/'06_limited_data_queue.parquet')}' (FORMAT PARQUET)")
display(con.execute(f"SELECT primary_rank,ein,peer_group,available_indicator_count,ROUND(adverse_peer_index,4) adverse_score,ROUND(peer_anomaly_index,4) original_score,original_anomaly_rank,isolation_anomaly_score FROM read_parquet('{sqlpath(qpath)}') WHERE review_cohort='primary_review' ORDER BY primary_rank LIMIT 20").df())


## 7. Quality checks and retrospective diagnostics

Check population coverage, rank uniqueness, top-ranked indicator availability, directional contributions, and overlap between the original and revised top 500. Diversion labels are used **only for retrospective description**, not to tune this unsupervised score.

In [ ]:
qpath=OUT/'06_2024_form990_anomaly_queue.parquet'
qa=con.execute(f"""
SELECT COUNT(*) n, COUNT(DISTINCT OBJECTID) distinct_ids,
       COUNT(*) FILTER (WHERE review_cohort='primary_review') primary_n,
       COUNT(*) FILTER (WHERE review_cohort='limited_data') limited_n,
       COUNT(*) FILTER (WHERE available_indicator_count<4 AND review_cohort='primary_review') invalid_primary,
       COUNT(*) FILTER (WHERE available_indicator_count>=4 AND review_cohort='limited_data') invalid_limited,
       MIN(adverse_peer_index) min_score,MAX(adverse_peer_index) max_score
FROM read_parquet('{sqlpath(qpath)}')
""").df()
display(qa)
assert qa.n.iloc[0]==qa.distinct_ids.iloc[0]
assert qa.n.iloc[0]==con.execute('SELECT COUNT(*) FROM base990').fetchone()[0]
assert qa.invalid_primary.iloc[0]==0 and qa.invalid_limited.iloc[0]==0
assert 0<=qa.min_score.iloc[0]<=qa.max_score.iloc[0]<=1.000001
rank_qa=con.execute(f"""SELECT COUNT(*) n, COUNT(DISTINCT primary_rank) unique_ranks,
MIN(primary_rank) first_rank,MAX(primary_rank) last_rank,
MIN(available_indicator_count) min_indicators
FROM read_parquet('{sqlpath(qpath)}') WHERE review_cohort='primary_review'""").df()
display(rank_qa)
assert rank_qa.n.iloc[0]==rank_qa.unique_ranks.iloc[0]
assert rank_qa.first_rank.iloc[0]==1 and rank_qa.last_rank.iloc[0]==rank_qa.n.iloc[0]
assert rank_qa.min_indicators.iloc[0]>=4
comparison=con.execute(f"""
SELECT COUNT(*) FILTER (WHERE original_anomaly_rank<=500 AND primary_rank<=500) overlap_top500,
       COUNT(*) FILTER (WHERE original_anomaly_rank<=500) original_top500,
       COUNT(*) FILTER (WHERE primary_rank<=500) revised_top500,
       AVG(available_indicator_count) FILTER (WHERE original_anomaly_rank<=100) original_top100_avg_indicators,
       AVG(available_indicator_count) FILTER (WHERE primary_rank<=100) revised_top100_avg_indicators
FROM read_parquet('{sqlpath(qpath)}')""").df()
display(comparison)
labels=con.execute(f"""
SELECT CASE WHEN q.primary_rank<=100 THEN 'top_100'
            WHEN q.primary_rank<=500 THEN 'rank_101_500'
            WHEN q.primary_rank<=1000 THEN 'rank_501_1000'
            WHEN q.review_cohort='limited_data' THEN 'limited_data'
            ELSE 'remaining_primary' END rank_band,
       COUNT(*) filings,
       COUNT(*) FILTER (WHERE f.fraud_label=1) reported_diversions
FROM read_parquet('{sqlpath(qpath)}') q
JOIN features f ON q.OBJECTID=f.OBJECTID AND f.tax_year=2024
GROUP BY 1 ORDER BY 1""").df()
display(labels)
# Show leading adverse contributions for human review, not fraud accusations.
leaders=con.execute(f"""SELECT primary_rank,ein,available_indicator_count,
 adverse_expense_revenue_ratio,adverse_liabilities_assets_ratio,adverse_net_margin,
 adverse_revenue_growth_3yr,adverse_asset_growth_3yr,
 adverse_revenue_volatility_3yr,adverse_board_independence
 FROM read_parquet('{sqlpath(qpath)}') WHERE primary_rank<=20 ORDER BY primary_rank""").df()
display(leaders)
qa.to_csv(OUT/'07_quality_checks.csv',index=False)
rank_qa.to_csv(OUT/'07_rank_integrity.csv',index=False)
comparison.to_csv(OUT/'07_ranking_comparison.csv',index=False)
labels.to_csv(OUT/'07_retrospective_label_diagnostics.csv',index=False)
leaders.to_csv(OUT/'07_top20_directional_contributions.csv',index=False)
meta={'notebook':'10 revised','scope':'2024 full Form 990 eligible cohort from 09D',
      'ranking':'directional adverse peer tail, fixed 7-indicator denominator; heuristic, not fraud probability',
      'minimum_indicators_primary':4,'limited_data_retained':True,
      'original_two_sided_score_preserved':True,
      'peer_groups':'2024 revenue bands','min_peer_size':MIN_PEER_SIZE,
      'isolation_forest':USE_ISOLATION_FOREST,'fit_sample':FIT_SAMPLE if USE_ISOLATION_FOREST else 0,
      'seed':SEED,'source':str(INPUT),'cohorts':str(COHORTS),
      'limitations':['self-reported diversion is not confirmed fraud',
                     'directional assumptions are not validated fraud predictors',
                     '2024 retrospective diagnostics are not independent validation',
                     'peer groups do not yet account for nonprofit subsector',
                     '2024 990-EZ excluded from scoring; not presumed low risk',
                     'outliers may be legitimate; human review required']}
(OUT/'10_manifest.json').write_text(json.dumps(meta,indent=2))
print('Saved revised Notebook 10 outputs to',OUT)


## Interpretation and Next Steps

### 1. Overview

Notebook 10 developed a financial and governance anomaly-detection framework using 2024 IRS Form 990 filings.

Unlike the supervised models developed in earlier notebooks, this approach does not attempt to predict self-reported diversions of assets. Instead, it identifies organizations exhibiting unusual financial and governance patterns that may warrant further investigation.

The analysis includes:

- Revenue-based peer-group benchmarking.
- Financial and governance anomaly indicators.
- Directional scoring of potentially adverse financial patterns.
- Isolation Forest anomaly detection.
- Data-coverage requirements for investigation eligibility.
- An explainable, ranked investigation candidate queue.

The objective is to identify potential investigative leads, not to establish that an organization has committed fraud.

### 2. Dataset and Scoring Population

The analysis included **355,973 Form 990 filings from tax year 2024**.

The revised scoring methodology separated organizations into two groups:

| Population | Filings |
|---|---:|
| Primary review population (at least 4 of 7 indicators) | 352,416 |
| Limited-data population (fewer than 4 indicators) | 3,557 |
| **Total** | **355,973** |

Organizations with insufficient data were retained for separate review rather than excluded from the dataset.

This distinction prevents organizations with only a small number of available indicators from automatically receiving disproportionately high investigation rankings.

### 3. Revised Anomaly-Ranking Methodology

The original peer-anomaly index treated unusually high and unusually low values similarly and averaged whichever indicators were available.

This created two problems:

1. Organizations with only one or two extreme measurements could rank above organizations exhibiting multiple concerning patterns.
2. Unusual but potentially favorable financial characteristics could increase an organization's risk ranking.

The revised methodology addressed these issues by:

- Requiring at least four available indicators for the primary investigation queue.
- Introducing directional scoring for potentially adverse financial and governance patterns.
- Accounting for indicator availability when constructing rankings.
- Retaining individual indicator contributions to support interpretability.
- Preserving Isolation Forest as a separate, unsupervised anomaly signal.

These changes improve ranking consistency and explainability but do not independently establish that the new rankings are better at identifying fraud.

### 4. Results and Ranking Diagnostics

The revised methodology produced several meaningful changes.

| Metric | Result |
|---|---:|
| Total 2024 Form 990 filings | 355,973 |
| Primary review population | 352,416 |
| Limited-data population | 3,557 |
| Average available indicators among top 100 | 7.00 |
| Organizations shared between original and revised top 500 | 74 |
| Reported asset diversions in revised top 100 | 0 |
| Reported asset diversions in revised top 500 | 1 |
| Reported asset diversions in revised top 1,000 | 4 |

The revised top-ranked organizations generally have more complete financial and governance information than those identified by the original scoring methodology.

However, the substantial change in top-ranked organizations demonstrates that the results are sensitive to how anomaly indicators are defined and combined.

The low overlap with reported asset diversions should not automatically be interpreted as poor fraud-detection performance because self-reported diversions are not a comprehensive measure of fraudulent activity.

### 5. Remaining Limitations

Several methodological limitations remain:

- **No confirmed fraud labels:** Statistical anomalies are not evidence of fraudulent conduct.
- **Peer-group definition:** Organizations are currently grouped by revenue size, without nonprofit subsector or mission classifications.
- **Correlated indicators:** Expense-to-revenue ratio and net margin may represent overlapping financial information, potentially creating double-counting.
- **Scoring assumptions:** Indicator directions, coverage thresholds, and weighting rules are analytical choices that have not been validated against independent investigative outcomes.
- **Filing-type coverage:** The current analysis focuses on full Form 990 filings. Form 990-EZ organizations require a separate assessment of available financial indicators.
- **Point-in-time limitations:** The current rankings are retrospective and have not been validated as prospective investigative signals.

### 6. Next Steps: Notebook 11

**Notebook 11 — Fraud-Risk Signal Integration and Ranking Diagnostics**

Notebook 11 will:

1. Compare the revised peer-anomaly ranking with the existing supervised diversion-disclosure model.
2. Examine correlations between financial indicators and reduce potential double-counting.
3. Evaluate how much overlap exists between organizations identified by different modeling approaches.
4. Develop an exploratory combined ranking using supervised and unsupervised signals.
5. Assess ranking sensitivity to different signal weights.
6. Generate a consolidated dataset containing individual scores, rankings, and explanatory indicators.

The combined ranking will remain exploratory until its investigative usefulness can be evaluated against stronger evidence or investigator feedback.

### Final Conclusion

Notebook 10 established an unsupervised anomaly-detection framework for identifying nonprofit organizations with potentially concerning financial and governance patterns.

The revised methodology improved data-coverage requirements, directional scoring, and interpretability while preserving separate peer-based and Isolation Forest signals.

**Decision:** Retain the revised Notebook 10 as the anomaly-detection baseline and proceed to Notebook 11 to evaluate how these signals can be integrated with the existing supervised models.

The project's ultimate objective remains identifying potentially suspicious nonprofit activity that could warrant further review by a criminal investigations team, rather than predicting self-reported diversion disclosures alone.